In [7]:
import os
import json
import subprocess
import napari

In [10]:
# List of input directories
input_dirs = [
    r"D:\SEM tiles\sample 1\denoised",
    r"D:\SEM tiles\sample 2\denoised",
    r"D:\SEM tiles\sample 3\denoised"
    
]

# List of file prefixes,'00
file_prefixes = ['00001', '00001','00001']

subfolder_name = "stitched"

# Create stitched subfolders for each input directory
output_dirs = []

for directory in input_dirs:
    subfolder_path = os.path.join(directory, subfolder_name)

    # Create the subfolder if it doesn't exist
    if not os.path.exists(subfolder_path):
        os.makedirs(subfolder_path)
        print(f"Subfolder created: {subfolder_path}")
    else:
        print(f"Subfolder already exists: {subfolder_path}")

    # Store the full output paths in a list
    output_dirs.append(subfolder_path)



Subfolder already exists: D:\SEM tiles\sample 1\denoised\stitched
Subfolder already exists: D:\SEM tiles\sample 2\denoised\stitched
Subfolder already exists: D:\SEM tiles\sample 3\denoised\stitched


In [11]:
# Validated stitch parameters (CLEM sample dataset, 2026-09):
#   overlap=0.20 for all scans (spacing ~1024x768 px on 1280x960 tiles)
#   --filter-sigma 15 (fs 8 also tested, equivalent)
#   --maximum-shift 150  <- units are MICROMETERS (150 um = 300 px at placeholder pixel_size 0.5)
#   pixel_size=0.5 is a PLACEHOLDER; replace with true nm/px for correct physical metadata
# Pattern gotchas: the scan ID (00001 etc.) is a LITERAL in the pattern, not a wildcard;
#   X/Y coordinates are 1-indexed, zero-padded to 3 digits (X001, Y001)
# Process each file
output_files = []

for file_prefix, input_dir, output_dir in zip(file_prefixes, input_dirs, output_dirs):
    output_file = os.path.join(output_dir, f"{file_prefix}_ashlar.ome.tif")

    fileseries_args = [
        f"filepattern|{input_dir}|pattern={file_prefix} X{{col:03}} Y{{row:03}}.tif|overlap=0.20|pixel_size=0.5"
    ]

    command = [
        'ashlar',
        *fileseries_args,
        '--output', output_file,
        '--maximum-shift', '150',
        '--align-channel', '0',
        '--filter-sigma', '15'
    ]

    try:
        result = subprocess.run(command, check=True, capture_output=True, text=True)
        print(f"Successfully processed: {output_file}")
        output_files.append(output_file)
    except subprocess.CalledProcessError as e:
        print(f"\n--- Error processing: {output_file} ---")
        print(f"Exit code: {e.returncode}")
        print("Command:", ' '.join(e.cmd))
        print("STDOUT:\n", e.stdout)
        print("STDERR:\n", e.stderr)


Successfully processed: D:\SEM tiles\sample 1\denoised\stitched\00001_ashlar.ome.tif
Successfully processed: D:\SEM tiles\sample 2\denoised\stitched\00001_ashlar.ome.tif
Successfully processed: D:\SEM tiles\sample 3\denoised\stitched\00001_ashlar.ome.tif


In [5]:
from PIL import Image
import os
import glob

input_dir = r"D:\SEM tiles\sample 4\denoised"

files = glob.glob(os.path.join(input_dir, "00001*.tif"))

sizes = {}

for file in files:
    try:
        with Image.open(file) as img:
            size = img.size

        if size not in sizes:
            sizes[size] = []

        sizes[size].append(os.path.basename(file))

    except Exception as e:
        print(f"Error reading {file}: {e}")

print("\nImage size groups:\n")

for size, filenames in sizes.items():
    print(f"{size}: {len(filenames)} files")

    for f in filenames[:5]:
        print(f"   {f}")

    if len(filenames) > 5:
        print("   ...")


Image size groups:

(1280, 960): 1119 files
   00001 X001 Y001.tif
   00001 X001 Y002.tif
   00001 X001 Y003.tif
   00001 X001 Y004.tif
   00001 X001 Y005.tif
   ...
(2560, 1920): 1 files
   00001 X015 Y030.tif


## Y-boundary duplication audit (final; supersedes the earlier extent table)

**Method** (`final_scan.py`): for every row boundary x column, correlate the upper tile's bottom band
`[400:960]` with the lower tile's top band `[0:400]` (2x downsampled, whitened + Hann window, FFT NCC),
searching only the duplication zone (|dx| <= 120 px, dy in [-300, +260] px, well short of the ~357 px
nominal-overlap offset), then verify with plain Pearson r over the exact overlap window plus a
peak-specificity check (r must drop >= 0.2 at dy +/- 80). Positive control: 00003 X004 Y011->Y012 gives
r = 0.982 at (dy=+176, dx=+21) with r ~ 0.1 at all other offsets.

**Root cause**: stage-settling slip at row changes. The next row's top ~100-260 px (~50-130 um at the
placeholder 0.5 um/px) re-images the previous row's bottom strip as a pixel-accurate copy (r = 0.65-0.98);
ashlar then blends both copies, producing doubled/ghost structures at the seams. Irreparable by
stitching - affected tiles must be re-acquired.

**Confirmed duplicated boundaries** (dy, dx in px, plain r):

- 00003 Y002->Y003: c2 (+198,-86, .86), c3 (+206,-104, .85), c4 (+214,-110, .89), c5 (+208,-84, .80)
- 00003 Y008->Y009: c2 (+162,-90, .87), c3 (+174,-104, .95), c4 (+166,-54, .93)
- 00003 Y011->Y012 (whole row): c1 (+158,-120, .88), c2 (+168,+94, .97), c3 (+166,+88, .96), c4 (+172,+20, .93), c5 (+158,+72, .92)
- 00005 Y002->Y003: c6 (+210,-44, .78)
- 00005 Y005->Y006: c5 (+260,-120, .65)
- 00005 Y006->Y007: c1 (+220,-60, .96), c3 (+246,-120, .90), c5 (+220,-78, .89)
- 00005 Y009->Y010: c1 (+200,-4, .95), c2 (+176,-14, .87), c3 (+178,+78, .98), c4 (+188,+10, .93), c6 (+190,-4, .90), c7 (+184,+24, .96), c8 (+174,-50, .89); c5 partial (r=.59)
- 00005 Y011->Y012: c3 (+140,+44, .77), c4 (+184,-44, .94), c5 (+164,-12, .78), c6 (+150,-20, .84), c8 (+162,+26, .95), c9 (+158,+10, .87)

**Corrections vs the earlier report**:

- 00005 Y013->Y014 has **no** duplicates: the bands are blank (std 0.6-1.0). Earlier hits there were
  artifacts of broken lag unwrapping plus a comparison image rendered at the wrong offset.
- 00003 Y009->Y010 c6, 00005 Y003->Y004 c7 and Y006->Y007 c4 downgraded (plain r <= 0.43, not
  pixel-accurate copies).
- 00001 is clean except one ambiguous corner candidate (X001 Y003->Y004, r ~ 0.75 spread across many
  offsets = repetitive texture, not a sharp duplicate).

**Tooling note**: `extent_scan.py` had two lag-unwrapping bugs (dy subtracted half the FFT period; dx
never unwrapped negative lags), so its earlier dy/dx prints - and any conclusions drawn from them - were
unreliable. `final_scan.py` fixes both and is the authoritative tool. Also: 00005 bottom row Y015 is an
empty field (std 0.6-0.7); thin streaks there are scan-line noise. The v5 stitches remain the deliverable;
the v6 re-stitch (overlap=0.4) is invalid (nominal-grid shift).

## Red-box annotation cross-check (2026-09-28)

User annotated the visible duplicate structures directly on the v5 mosaics
(`stitched/00003_ashlar_v5.ome red ROI.tif`, `00005_... red ROI.tif`; RGB, canvas identical to v5 so
coordinates map 1:1). 20 red boxes detected (10 per image, >= 100 px each) and mapped to the grid
(steps spx/spv = 1024.0/752.5 for 00003, 1074.0/756.9 for 00005).

- All 20 boxes sit on row boundaries independently confirmed as duplicated by final_scan.py
  (00003: Y002/Y008/Y011; 00005: Y002/Y009/Y011) - the "horizontal lines" the user saw are the dup seams.
- 16/20 boxes fall entirely inside the confirmed column ranges; the small spills (00003 Y002 c1,
  Y008 c5, 00005 Y002 c5) are box width, not confirmed dup columns.
- One box (00005 Y002->Y003 centered on col 4) is NOT corroborated: the dup there is limited to c6.
- User confirmation: boxes surround the duplicated structures; most are offset vertically, a couple
  also horizontally - matching the measured dx up to +/-120 px alongside dy +140-260 px.

### 00001 refinement

The X001 Y003->Y004 candidate is a diagonal correlation ridge: the top 5 peaks all lie along
dy + dx ~ 260 (dy 120-176, dx 92-130, q 0.71-0.74), collapsing off-ridge (r ~ 0.1 at dy +/- 80 off
the ridge). That is repetitive diagonal-line texture, not a pixel-accurate copy - the original
"texture" call stands. (The dy +/- 80 specificity check alone can pass on diagonal ridges; the scan's
CONFIRMED verdict there was a false positive.) 00001 remains clean.

### Slip-mechanism probe (00003 c2)

At a dup boundary the ONLY strong alignment is the dup offset itself (Y002->Y003: dy +198 dx -86,
r = 0.86; no secondary peak near the nominal ~+352). At the following boundary (Y003->Y004) no
alignment exists anywhere in +/-360 px (band q ~ 0.22, plain r < 0.08), and whole-tile 4x-downsampled
correlations are weak everywhere (q ~ 0.33). The re-scanned strip is baked into the raw tiles; simple
realignment is not obviously fixable - the practical test is the patched restitcher (next cell).

## Patched restitcher (v7) - results (2026-09-28)

The "practical test" promised above is done. `restitch.py` (repo root) patches
Ashlar's registration instead of its parameters:

- `register_padded` replaces `ashlar.utils.register`: whiten + Hann window, FFT
  padded to 2x tile size (no wraparound lags), shift search hard-limited to
  +/-300 px, 10x cubic sub-pixel refinement.
- `patch_register_pair` replaces `EdgeAligner.register_pair`: multi-scale
  coarse-to-fine registration (intersection size up to 10% of tile size, cap
  50%), min-error scale selection, final error = whitened NCC over the actual
  overlap.
- Runs with the validated v3 parameters (overlap 0.20, filter-sigma 15,
  maximum-shift 150, align-channel 0): `python restitch.py <scan> <out> [overlap]`.

v7 outputs exist for all four scans (`<scan>_ashlar_v7.ome.tif`). Pyramids,
dtype and range verified; canvases within ~1% of v3 (0.20-overlap geometry
preserved - unlike the invalid v6). Aligned NCC vs v3: 00001 = 0.971,
00004 = 0.918; 00003/00005 reviewed visually and confirmed. v7 is the current
deliverable for all four scans; full V1->V7 history in `STITCHING_VERSIONS.md`.
The dup strips themselves remain baked into the raw tiles - re-acquisition is
still the only fix where pixel-accurate duplicates exist.